# Project 4: Chord Recognition Using Symbolic Data

This is the demo notebook for <i> Project 4: Chord Recognition Using Symbolic Data </i>, containing the MIDI chord-estimation pipeline used to generate labels for the symbolic data.

<b>Is it viable to generate datasets of chord labels using only symbolic (MIDI) data, good enough for the training and evaluation of existing chord recognizers?</b> <br>
###The structure of the notebook is as follows:

1. Establish Drive, initialize dataset and load data
2. Create track segments containing active notes <br>
  a. Segment multitracks by beat and measure <br>
  b. Store MIDI note data for each segment <br>
3. Label each segment with a chord estimate <br>
  a. Calculate Pitch Class Profile scores for segments <br>
  b. Find best estimate of chord based on similarity to chord templates <br>
4. Evaluation <br>
  a. Error analysis <br>
  b. Benchmarking an existing dataset using CREMA

Add the utils.py file to your files folder in order to properly import the functions contained within. This is notebook is just a high-level overview of the processing steps.


##Install packages and set up Drive as the dataset home

In [ ]:
# run in base_environment
!pip install pretty_midi mir_eval mirdata pyfluidsynth

# Packages
import pretty_midi
import mirdata
import mir_eval
import numpy as np
import pandas as pd
from pathlib import Path

# Our functions
import utils as u

# For plotting
import mir_eval.display
import librosa.display
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# For audio display
from IPython.display import Audio

## 1. Initialize dataset loader, download and load data using mirdata





For this project we will use SLAKH datset, a synthesized version of the LAKH dataset:
<blockquote>
Manilow, Ethan, Gordon Wichern, Prem Seetharaman, and Jonathan Le Roux. "Cutting music source separation some Slakh: A dataset to study the impact of training data quality and quantity." In 2019 IEEE Workshop on Applications of Signal Processing to Audio and Acoustics (WASPAA), pp. 45-49. IEEE, 2019.
</blockquote>

The full SLAKH dataset consists of 2100 multitrack objects with matching synthesized audio. Each multitrack contains track objects for each individual instrument, and each track object stores information about note events per instrument. All MIDI information is stored using <i> pretty_midi </i> structures, including Notes and Instruments.

<blockquote>
Colin Raffel and Daniel P. W. Ellis. Intuitive Analysis, Creation and Manipulation of MIDI Data with pretty_midi. In Proceedings of the 15th International Conference on Music Information Retrieval Late Breaking and Demo Papers, 2014.
</blockquote>

###The dataset will be installed in your drive:

In [ ]:
# Mount Drive to work with dataset
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
# Set location and version of dataset and load data
data_home = '/content/drive/MyDrive'
dataset_name = 'slakh'
dataset_version = 'baby' # use 'baby' for prototyping, default is 2100-redux
dataset = u.load_data(dataset_name, data_home=data_home, dataset_version=dataset_version)

# Run the following line once to download
#dataset.download()

# Download the index for mirdata to load
#dataset.download(partial_download=['index'])

# Uncomment the following line and run to validate
#dataset.validate()

### Sample a random multitrack and generate audio:
This ensures the data has loaded properly.

In [ ]:
# Sample a track and check Audio
example = dataset.choice_multitrack()
print(example.mtrack_id)

# listen to Dataset audio
Audio(example.audio[0], rate=example.audio[1])

# Listen to Synthesized Audio
synth = example.midi.synthesize(fs=44100)
Audio(synth, rate=44100)

# 2. Create Track segments containing active notes

## 2a: Segment the MIDI files by beats and downbeats

utils.beat_times calls the get_downbeats() and get_beats() functions from <i>pretty_midi</i> and returns timed divisions:

In [ ]:
# Load all multitracks
data = dataset.load_multitracks()

# Get downbeats and beats for dataset using PrettyMIDI function
downbeats = u.beat_times(data, division='downbeat')
beats = u.beat_times(data, division='beat')

## 2b: Using beat and downbeat times, find segments of MIDI and store active notes
utils.segment_midi() takes the dataset dictionary and the dictionary of beat times, storing a collection of values ('start', 'end', 'notes') for each segment of each track. All three dictionaries are keyed by mtrack_id.

In [ ]:
# Get a dictionary of segment information for the dataset
beat_segments = u.segment_midi(data, beats)
dbeat_segments = u.segment_midi(data, downbeats)

#### Create click track to check alignment of audio to extracted midi segments

In [ ]:
def create_click_waveform(freq, fs=44100, duration=0.05):
    t = np.linspace(0, duration, int(fs * duration))
    # generate sine wave
    click = np.sin(2 * np.pi * freq * t)
    # apply exponential decay (the envelope) so it sounds like a 'click'
    envelope = np.exp(-t / (0.01 * duration))
    return click * envelope

regular_waveform = create_click_waveform(440)

In [ ]:
# Load multitrack
audio = dataset.choice_multitrack()
track_id = audio.mtrack_id
# Store multitrack audio data
audio_data, fs = audio.audio

# Generate clicks at the beat times of the track
beat_clicks = mir_eval.sonify.clicks(beats[track_id], fs=fs, length=len(audio_data), click=regular_waveform)
# At the downbeats
downbeat_clicks = mir_eval.sonify.clicks(downbeats[track_id], fs=fs, length=len(audio_data))

# Play the clicks alongside the original audio
Audio(audio_data + beat_clicks + downbeat_clicks, rate=fs)

# 3. Label each segment with a chord estimate

##3a: Calculate the weighted Pitch Class Profile scores for each segment of each multitrack

Pitch Class Profile score is calculated as the sum over octaves of the velocity * relative duration of each note active during the segment. Formula acquired from the CASSETTE model:
<blockquote>
Odekerken, Daphne, Hendrik Vincent Koops, and Anja Volk. 2021. “Improving Audio Chord Estimation by Alignment and Integration of Crowd-sourced Symbolic Music”. Transactions of the International Society for Music Information Retrieval 4 (1): 141–155. https://doi.org/10.5334/tismir.81.
</blockquote>


In [ ]:
beat_pcp = u.weighted_pitch_class(beat_segments)
dbeat_pcp = u.weighted_pitch_class(dbeat_segments)

## 3b: Define chord templates and find best estimate for each pitch class profile

The CASSETTE paper used only major and minor triad templates, along with a No Chord template. Templates for chord patterns defined below, utils.get_all_templates rolls the pattern over all chord roots with the additional 'N' to get a full vocabulary.

In [ ]:
# define chord templates
# start with basic maj/min + 'N' chords
CHORD_PATTERNS = {
    'maj': [1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0],
    'min': [1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0],
    #'maj7': [1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1],
    #'min7': [1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0],
    #'7': [1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0]
}

templates = u.get_all_templates(CHORD_PATTERNS)

In [ ]:
enharmonic_map = {
    'Db': 'C#',
    'Eb': 'D#',
    'Gb': 'F#',
    'Ab': 'G#',
    'Bb': 'A#',
    'Cb': 'B',
    'Fb': 'E',
    'E#': 'F',
    'B#': 'C'
}

### Calculate and compare similarity scores for each template and choose the best classification.
<i>classify_chord()</i> from utils.py uses function <i>calc_similarity()</i> to compare similarity scores for each template and store the best chord estimate + corresponding similarity score for each segment of a multitrack.

In [ ]:
# Create dictionaries to store estimates for all multitracks
# Each key will have chord array and similarity score array (n mtracks x (2 x m segments))
beat_estimates = {}
dbeat_estimates = {}

for id, pcp in beat_pcp.items():
    beat_estimates[id] = u.classify_chord(pcp, templates)

for id, pcp in dbeat_pcp.items():
    dbeat_estimates[id] = u.classify_chord(pcp, templates)

Test that the estimates are working on a random track, and check that there is a chord estimate for every similarity score.

In [ ]:
# Test that estimates are working
db_chordlen = len(dbeat_estimates['Track00016'][0])
db_simlen = len(dbeat_estimates['Track00016'][1])
b_chordlen = len(beat_estimates['Track00016'][0])
b_simlen = len(beat_estimates['Track00016'][1])

# Test length of arrays
if  db_chordlen == db_simlen:
  print(f"Downbeat chords and similarities same length: {db_chordlen}")
if  b_chordlen == b_simlen:
  print(f"Beat chords and similarities same length: {b_chordlen}")

print(dbeat_estimates['Track00016'][0])
print(beat_estimates['Track00016'][0])

# 4: Evaluation (assuming CREMA outputs calculated)

Chord estimates and respective similarity scores are stored in dictionaries <i>dbeat_estimates</i> and <i>beat_estimates</i> at indices [0] and [1], respectively.

CREMA estimates generated and saved using process in separate process_crema.ipynb file.

In [ ]:
# Save cassette outputs to a csv file. Crema files are stored in the same way for ease of access.
for track in dataset.mtrack_ids:
    u.save_cassette_csv(beats, beat_estimates, track, downbeats=False)
    u.save_cassette_csv(downbeats, dbeat_estimates, track, downbeats=True)

#### a. Visualize chord annotations between Cassette beat/downbeat labels and CREMA estimations

In [ ]:
def get_chord_colors(all_labels):
    """Auto-generate a color map for any set of chord labels."""
    cmap = plt.cm.get_cmap('tab20', len(all_labels))
    return {label: cmap(i) for i, label in enumerate(sorted(all_labels))}

def plot_chord_comparison(track_id, chord_colors=None):
    """
    Plot a chord comparison visualization for a given track,
    comparing CASSETTE (beat-level) vs CREMA estimates over time.
    """
    # Load data from csv files for beats, downbeats, and CREMA
    downbeats_df = pd.read_csv(f'./output/cassette_dbeats/{track_id}.csv')
    beats_df = pd.read_csv(f'./output/cassette_beats/{track_id}.csv')
    #crema_df    = pd.read_csv(f'./output/crema/{track_id}.csv')

    #crema_df         = u.process_crema_df(crema_df, enharmonic_map)
    #crema_beatwise_df = u.get_beatwise_crema(crema_df, cassette_df, u.majority_label)

    # --- Build annotation dict: label -> list of (start, end, chord) ---
    # Adjust column names below if yours differ (e.g. 'onset'/'offset' instead of 'start'/'end')
    annotations = {
        "DOWNBEATS":     list(zip(downbeats_df['start_time'], downbeats_df['end_time'], downbeats_df['value'])),
        "BEATS":  list(zip(beats_df['start_time'], beats_df['end_time'], beats_df['value'])),
        #"CREMA": list(zip(crema_beatwise_df['start_time'], crema_beatwise_df['end_time'], crema_beatwise_df['value']))
    }

    # Auto-build color map
    if chord_colors is None:
        all_labels = set(downbeats_df['value']) | set(beats_df['value'])
        chord_colors = get_chord_colors(all_labels)

    # Build Plots
    fig, axes = plt.subplots(
        nrows=len(annotations),
        figsize=(18, len(annotations) * 1.1 + 1.8),
        sharex=True
    )
    if len(annotations) == 1:
        axes = [axes]

    row_height = 0.7
    used_labels = set()

    for ax, (label, segments) in zip(axes, annotations.items()):
        for (start, end, chord) in segments:
            color = chord_colors.get(chord, "#CCCCCC")
            ax.barh(y=0, width=end - start, left=start,
                    height=row_height, color=color, edgecolor="none")
            used_labels.add(chord)
        ax.set_yticks([0])
        ax.set_yticklabels([label], fontsize=11, fontweight='bold')
        ax.set_ylim(-0.5, 0.5)
        ax.set_facecolor("white")
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.tick_params(left=False, bottom=False)

    axes[-1].set_xlabel("Time (s)", fontsize=11)

    # Legend (for relevant chords)
    legend_patches = [
        mpatches.Patch(color=chord_colors[chord], label=chord)
        for chord in sorted(used_labels)
    ]
    fig.legend(
        handles=legend_patches,
        loc="lower center",
        ncol=min(len(used_labels), 10),
        fontsize=8,
        frameon=False,
        bbox_to_anchor=(0.5, -0.04)
    )

    fig.patch.set_facecolor("white")
    plt.suptitle(f"Chord Comparison: {track_id}", fontsize=13,
                 color='black', y=1.01)
    plt.tight_layout()
    plt.savefig(f"./output/chord_viz_{track_id}.png",
                dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show()


# Plot chord comparison on any track
plot_chord_comparison('Track00016')
audio, sr = data['Track00016'].audio
Audio(audio, rate=sr)


#### B. Manual scoring:

Evaluation of MIDI-generated chord labels done by scoring recall of CREMA given the multi-track audio, treating labels as ground truth.

To compare the output of crema to cassette, utils.process_crema_df and utils.get_beatwise_crema time aligns the outputs and flattens enharmonic equivalencies.

In [ ]:
# Evaluate all tracks
accuracy_list = []
accuracy_inv_list = []
accuracy_triad_list = []

for track in dataset.mtrack_ids:
    # load relevant cassette csv
    cassette_df = pd.read_csv(f'./output/cassette/{track}.csv')
    # load relevant crema csv
    crema_df = pd.read_csv(f'./output/crema/{track}.csv')

    # process crema outputs to align with cassette's for comparison
    crema_df = u.process_crema_df(crema_df, enharmonic_map)
    crema_beatwise_df = u.get_beatwise_crema(crema_df, cassette_df, u.majority_label)

    # manual evaluation
    accuracy, accuracy_no_inv, accuracy_triad = u.manual_chord_evaluation(crema_beatwise_df, cassette_df)
    accuracy_list.append(accuracy)
    accuracy_inv_list.append(accuracy_no_inv)
    accuracy_triad_list.append(accuracy_triad)

print(f"average accuracy: {np.mean(accuracy_list):.2f}%")
print(f'average accuracy without inversions: {np.mean(accuracy_list):.2f}%')
print(f"average accuracy triads: {np.mean(accuracy_triad_list):.2f}%")

#### C. Generating mir_eval standard scores for <i>root</i>, <i>third</i>, <i>triad</i>, <i>seventh</i>, etc. using mir_eval.chord.evaluate()

In [ ]:
# without normalization
scores = {}

for track in dataset.mtrack_ids:

    # load relevant cassette csv
    cassette_df = pd.read_csv(f'./output/cassette/{track}.csv')
    # load relevant crema csv
    crema_df = pd.read_csv(f'./output/crema/{track}.csv')

    # process crema outputs to align with cassette's for comparison
    crema_df = u.process_crema_df(crema_df, enharmonic_map)

    # get chord scores
    score = u.get_mir_chord_scores(crema_df, cassette_df, normalize_labels=False)

    scores[track] = score

results_df = pd.DataFrame(scores).T
final_scores = results_df.mean(numeric_only=True)
print(final_scores)

In [ ]:
# with normalization
scores_normalized = {}

for track in dataset.mtrack_ids:

    # load relevant cassette csv
    cassette_df = pd.read_csv(f'./output/cassette/{track}.csv')
    # load relevant crema csv
    crema_df = pd.read_csv(f'./output/crema/{track}.csv')

    # process crema outputs to align with cassette's for comparison
    crema_df = u.process_crema_df(crema_df, enharmonic_map)

    # get chord scores
    score = u.get_mir_chord_scores(crema_df, cassette_df, normalize_labels=True)

    scores_normalized[track] = score

results_normalized_df = pd.DataFrame(scores_normalized).T
final_scores_normalized = results_normalized_df.mean(numeric_only=True)
print(final_scores_normalized)

both manual and mir_eval pipelines show same hierarchy of agreement: root-level similarity is high, triad-level similarity is moderate, and extension-sensitive metrics are lowest, regardless of normalization choices

In [ ]:
# plot cm for 1 track
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# load relevant cassette csv
cassette_df = pd.read_csv(f'./output/cassette/Track00001.csv')
# load relevant crema csv
crema_df = pd.read_csv(f'./output/crema/Track00001.csv')

# process crema outputs to align with cassette's for comparison
crema_df = u.process_crema_df(crema_df, enharmonic_map)
crema_beatwise_df = u.get_beatwise_crema(crema_df, cassette_df, u.majority_label)

y_true = crema_beatwise_df['value']
y_pred = cassette_df['value']
labels = sorted(set(y_true) | set(y_pred))

cm = confusion_matrix(y_true, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
cm_df.div(cm_df.sum(axis=1), axis=0) # normalize

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_df,
    cmap="Blues",
    annot=False,
    linewidths=0.5
)

plt.xlabel("Predicted (Cassette)")
plt.ylabel("True (CREMA)")
plt.title("Chord Confusion Matrix (Cassette vs CREMA)")
plt.tight_layout()
plt.show()